<a href="https://colab.research.google.com/github/akshat280706/ML-Lab-Experiment/blob/main/241080009_akshat_ML_LAB3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded=files.upload()

Saving mushrooms.csv to mushrooms.csv


In [ ]:
import pandas as pd
import math
from sklearn.model_selection import train_test_split

data=pd.read_csv("mushrooms.csv")
target="class"

train_data, test_data = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data[target]
)

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
def entropy(data):
  counts=data[target].value_counts()
  total=len(data)
  e=0
  for count in counts:
    p=count/total
    e-=p*math.log2(p)
  return e


def information_gain(data, attribute):
  total_entropy=entropy(data)
  weighted_entropy=0

  for value in data[attribute].unique():
    subset=data[data[attribute]==value]
    weight=len(subset)/len(data)
    weighted_entropy+=weight*entropy(subset)
  return total_entropy-weighted_entropy


def gain_ratio(data,attribute):
  gain=information_gain(data,attribute)
  split_info=0

  for value in data[attribute].unique():
    subset=data[data[attribute]==value]
    p=len(subset)/len(data)

    split_info-=p*math.log2(p)

  if split_info==0:
    return 0
  return gain/split_info


def gini(data):
  counts=data[target].value_counts()
  g=1

  for count in counts:
    p=count/len(data)
    g-=p**2
  return g

def gini_gain(data,attribute):
  total=0
  for value in data[attribute].unique():
    subset=data[data[attribute]==value]
    weight=len(subset)/len(data)
    total+=weight*gini(subset)
  return total

def misclassification(data):
  counts=data[target].value_counts()
  maxi=max(counts)
  return 1-(maxi/len(data))

def misclassification_gain(data,attribute):
  total=0
  for value in data[attribute].unique():
    subset=data[data[attribute]==value]
    weight=len(subset)/len(data)
    total+=weight*misclassification(subset)
  return total

def best_attribute(data, attributes, method):
    scores = {}
    print("\nScores")

    for attribute in attributes:
        if method == "gain":
            score = information_gain(data, attribute)
        elif method == "ratio":
            score = gain_ratio(data, attribute)
        elif method == "gini":
            score = -gini_gain(data, attribute)
        else:
            score = -misclassification_gain(data, attribute)

        scores[attribute] = score
        print(attribute, "=", round(score, 5))
    return max(scores, key=scores.get)

def create_tree(data,attributes,methods):
  classes=data[target]
  if len(classes.unique())==1:
    return classes.iloc[0]
  if len(attributes)==0:
    return classes.mode()[0]

  best = best_attribute(data, attributes, method)
  print("\nselected:", best)
  tree = {best: {}}

  remaining_attribute = attributes.copy()
  remaining_attribute.remove(best)

  for value in data[best].unique():
    subset=data[data[best]==value]

    if len(subset)==0:
      tree[best][value]=classes.mode()[0]
    else:
      tree[best][value] = create_tree(
      subset,remaining_attribute, method

      )
  return tree


def predict(tree, row):
    if not isinstance(tree, dict):
        return tree

    root = list(tree.keys())[0]
    value = row[root]

    if value not in tree[root]:
        return None
    return predict(tree[root][value], row)


def print_rules(tree, rule=""):
  if not isinstance(tree, dict):
    print(rule,"then class= ", tree)
    return
  root=list(tree.keys())[0]
  for value in tree[root]:
    if rule=="":
      new_rule="if "+root+" = "+str(value)
    else:
      new_rule=rule+" and "+root+" = "+str(value)
    print_rules(tree[root][value],new_rule)


attributes = list(train_data.columns)
attributes.remove(target)
methods = {
    "gain": "Information Gain",
    "ratio": "Gain Ratio",
    "gini": "Gini Index",
    "misclassification": "Misclassification Error"
}

for method in methods:
    print(methods[method])

    tree = create_tree(
        train_data,
        attributes,
        method
    )



    print("\nroot node")
    print(list(tree.keys())[0])

    print("\nFollowing rules: ")
    print_rules(tree)

    predictions = []

    for _, row in test_data.iterrows():
        predictions.append(predict(tree, row))

    correct = sum(
        actual == predicted
        for actual, predicted in zip(
            test_data[target],
            predictions
        )
    )
    accuracy = correct / len(test_data)
    print("\nAccuracy =", round(accuracy * 100, 2), "%")

    actual = test_data[target]
    cm = confusion_matrix(actual, predictions)
    print("\nConfusion Matrix: ")
    print(cm)

Information Gain

Scores
cap-shape = 0.04837
cap-surface = 0.02648
cap-color = 0.03483
bruises = 0.19325
odor = 0.90531
gill-attachment = 0.01412
gill-spacing = 0.10135
gill-size = 0.22358
gill-color = 0.41618
stalk-shape = 0.00778
stalk-root = 0.1371
stalk-surface-above-ring = 0.28871
stalk-surface-below-ring = 0.27305
stalk-color-above-ring = 0.25596
stalk-color-below-ring = 0.2447
veil-type = 0.0
veil-color = 0.02291
ring-number = 0.03758
ring-type = 0.31931
spore-print-color = 0.48586
population = 0.20385
habitat = 0.15274

selected: odor

Scores
cap-shape = 0.04221
cap-surface = 0.01591
cap-color = 0.0939
bruises = 0.00113
gill-attachment = 0.00273
gill-spacing = 0.00648
gill-size = 0.0203
gill-color = 0.08836
stalk-shape = 0.06295
stalk-root = 0.02217
stalk-surface-above-ring = 0.02241
stalk-surface-below-ring = 0.04976
stalk-color-above-ring = 0.03439
stalk-color-below-ring = 0.05968
veil-type = 0.0
veil-color = 0.01126
ring-number = 0.02689
ring-type = 0.00072
spore-print-color